In [4]:
k = 32
dmodel = 64 * k
print(f'dmodel: {dmodel}')
n_layers = k
tt_params = 12 * n_layers * (dmodel ** 2)
emb_params = 50_000 * dmodel
model_params = emb_params + tt_params
print(f"model: {model_params / 1e9:.2f}B")

dmodel: 2048
model: 1.71B


In [24]:
batch_size = 256
seq_len = 1024
steps = 1000_000

tokens = batch_size * seq_len * steps
tokens = 1000e9
print(f"tokens: {tokens / 1e9:.2f}B")

print(f'tokens/params ratio:{tokens/model_params:.2f}')

tokens: 1000.00B
tokens/params ratio:583.77


In [26]:
gpus = 32
# time = 12 * (60 ** 2)
gpu_flops = 835 * 1e12  #H100
mfu = 0.3
# gpu_flops = 312 * 1e12  #A100
flops = 6 * model_params * tokens

time = (flops / (gpu_flops * gpus * mfu)) / 60 ** 2
print(f"time: {time:.2f}h")
print(f"time: {time / 24:.2f}d")

gpu_hours = time * gpus
print(f"GPU hours: {gpu_hours:.0f}")

time: 356.17h
time: 14.84d
GPU hours: 11397


In [8]:
current_steps = 2500
current_time = 4 * 60
total_steps = 50_000
time_total = (total_steps / current_steps) * current_time
print(f'time total: {time_total / (60 **2 ):.2f} h')

time total: 1.33 h


In [17]:
def calc_grid_flops(n_lrs_per_dmodel: int, dmodels: list, n_layers: int, tokens: int, n_plots: int):
    total_flops = 0
    flops_list = []
    for dmodel in dmodels:
        params = 12 * (dmodel ** 2) * n_layers
        flops = tokens * params * n_lrs_per_dmodel * 6
        flops_list.append(flops)
        total_flops += flops
    
    return total_flops * n_plots

In [18]:
grid_dense = {
    "n_lrs_per_dmodel": 5,
    "dmodels": [128, 256, 512, 768, 1024, 1536],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 2
}

grid_moe_new = {
    "n_lrs_per_dmodel": 1,
    "dmodels": [128, 256, 512, 768],
    "n_layers": 16,
    "tokens": 5 * 1e9,
    "n_plots": 1,
}

grid_moe_new_helios = {
    "n_lrs_per_dmodel": 5,
    "dmodels": [2048, 3584],
    "n_layers": 16,
    "tokens": 5 * 1e9,
    "n_plots": 3,
}

current_128 = {
    "n_lrs_per_dmodel": 5,
    "dmodels": [128],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 3,
}
current_256 = {
    "n_lrs_per_dmodel": 5,
    "dmodels": [256],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 3,
}
current_512 = {
    "n_lrs_per_dmodel": 4,
    "dmodels": [512],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 1,
}
current_1024 = {
    "n_lrs_per_dmodel": 1,
    "dmodels": [1024],
    "n_layers": 24,
    "tokens": 16 * 1e9,
    "n_plots": 1,
}


dense_flops = calc_grid_flops(**grid_dense)
print(f"{dense_flops / 1e18} * 1e18")

dmodels = [128, 256, 512, 768, 1024, 1536, 2048, 3584]
for dm in dmodels:
    grid_moe_new["dmodels"] = [dm]
    flops_dm = calc_grid_flops(**grid_moe_new)
    print(f"dmodel: {dm}\tflops:{flops_dm/1e18:.2f} * 1e18")


1200.4098048 * 1e18
dmodel: 128	flops:0.09 * 1e18
dmodel: 256	flops:0.38 * 1e18
dmodel: 512	flops:1.51 * 1e18
dmodel: 768	flops:3.40 * 1e18
dmodel: 1024	flops:6.04 * 1e18
dmodel: 1536	flops:13.59 * 1e18
dmodel: 2048	flops:24.16 * 1e18
dmodel: 3584	flops:73.99 * 1e18


In [19]:
small = 1.5 * 1e16
big = 4 * 1e18
print(big/small)

266.6666666666667


In [20]:
h100_flops = 835 * 1e12  #H100
a100_flops = 312 * 1e12  #A100
h100_flops_per_day = 24 * 60 * 60 * h100_flops
a100_flops_per_day = 24 * 60 * 60 * a100_flops
mfu = 0.1
print(f"{a100_flops_per_day * 8 * mfu / 1e18} * 1e18")
print(f"{h100_flops_per_day * 8 * mfu / 1e18} * 1e18")


21.56544 * 1e18
57.7152 * 1e18


In [21]:
2048/128

16.0